In [0]:
secret-id="ur secret id"
app-id="ur application id"
tenat-id="directory id or tenant id"

**Connection setup**

In [0]:


spark.conf.set("fs.azure.account.auth.type.salessource.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.salessource.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.salessource.dfs.core.windows.net","appid here")
spark.conf.set("fs.azure.account.oauth2.client.secret.salessource.dfs.core.windows.net", "secret here")
spark.conf.set("fs.azure.account.oauth2.client.endpoint.salessource.dfs.core.windows.net", "https://login.microsoftonline.com/tenant id here/oauth2/token")

dbutils.fs.ls("abfss://sales-source@salessource.dfs.core.windows.net")

[FileInfo(path='abfss://sales-source@salessource.dfs.core.windows.net/customers/', name='customers/', size=0, modificationTime=1780427623000),
 FileInfo(path='abfss://sales-source@salessource.dfs.core.windows.net/orders/', name='orders/', size=0, modificationTime=1780427647000),
 FileInfo(path='abfss://sales-source@salessource.dfs.core.windows.net/products/', name='products/', size=0, modificationTime=1780427636000),
 FileInfo(path='abfss://sales-source@salessource.dfs.core.windows.net/test/', name='test/', size=0, modificationTime=1781118811000)]

**Reading Files**

In [0]:
df=spark.read.format('csv')\
    .option("header","true")\
    .option("inferSchema","true")\
    .load("abfss://sales-source@salessource.dfs.core.windows.net/customers")

**Select transformation**

In [0]:
df=df.select("city","state","country","zip_code","customer_segment")

**Action Button**

In [0]:
df.display()

city,state,country,zip_code,customer_segment
New York,NY,USA,10001,Premium
Los Angeles,CA,USA,90001,Standard
Chicago,IL,USA,60601,Premium
Houston,TX,USA,77001,Standard
Phoenix,AZ,USA,85001,VIP
Philadelphia,PA,USA,19101,Standard
San Antonio,TX,USA,78201,Premium
San Diego,CA,USA,92101,VIP
Dallas,TX,USA,75201,Standard
San Jose,CA,USA,95101,Premium


For Enabling Adabtive Query Execution

In [0]:
spark.conf.set("spark.sql.adaptive.enabled","false")

In [0]:
spark.conf.get("spark.sql.adaptive.enabled"	)

'false'

**Importing all pyspark transformation functions**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Creating our own customized dataframe**

In [0]:
dataframe=[(1,'ali','ISB'),
           (2,'ali','LHR')]
schema=StructType([StructField("id",IntegerType(),True),
                   StructField("name",StringType(),True),StructField("city",StringType(),True)])
df2=spark.createDataFrame(dataframe,schema)


In [0]:
df2.display()

id,name,city
1,ali,ISB
2,ali,LHR


**Doing join**

In [0]:
df.join(df2,df.city==df2.city,"inner").display()

customer_id,first_name,last_name,email,phone,city,state,country,zip_code,customer_segment,registration_date,id,name,city


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

**aggregation on customer_id**

In [0]:
df.agg(count(col('customer_id'))).display()

count(customer_id)
15


**Checking key vault activated**

In [0]:
dbutils.secrets.get(scope="test-scope",key="app-id"	)

'[REDACTED]'

**Writing file as delta format to an external location like azure datalake **

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://sales-source@salessource.dfs.core.windows.net/customers_output")

In [0]:
%sql
CREATE DATABASE salesdb;

**Creating Table**

In [0]:
%sql
CREATE TABLE salesdb.sales(
    id INT,
    name STRING,
    age INT,
    gender STRING
)

In [0]:
%sql
INsERT INTO salesdb.sales (id,name,age,gender)
values (1,'John',25,'Male'),
(2,'Jane',30,'Female'),
(3,'Bob',40,'Male'),
(4,'Alice',35,'Female'),
(5,'Charlie',28,'Male')

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
SELECT * FROM salesdb.sales

id,name,age,gender
1,John,25,Male
2,Jane,30,Female
3,Bob,40,Male
4,Alice,35,Female
5,Charlie,28,Male


In [0]:
%sql
drop table salesdb.sales

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

name,url,comment
databricks_sales_dev,abfss://unity-catalog-storage@dbstorageahhzqj5ipen3w.dfs.core.windows.net/7405607761208239,null


In [0]:
df.write.format("delta")\
    .mode("overwrite"	)\
    .saveAsTable("salesdb.sales")

**Removing duplicates using reuseable function**


In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, concat_ws, to_timestamp

def dedup(df: DataFrame, col_list: list, reg_date: str) -> DataFrame:

    df.withColumn("dedup_key", concat(*col_list))

    window_spec = Window.partitionBy('dedup_key') \
                        .orderBy(to_timestamp(col(reg_date)).desc())

    df = df.withColumn("dedup_counts", row_number().over(window_spec))

    df = df.filter(col("dedup_counts") == 1) \
           .drop("dedup_counts")

    return df

In [0]:
df=dedup(df_sales,['customer_id'],'registration_date')


In [0]:
df.display()

customer_id,first_name,last_name,email,phone,city,state,country,zip_code,customer_segment,registration_date
C001,Alice,Johnson,alice.johnson@email.com,+1-555-0101,New York,NY,USA,10001,Premium,2021-03-15
C002,Bob,Smith,bob.smith@email.com,+1-555-0102,Los Angeles,CA,USA,90001,Standard,2020-06-22
C003,Carol,Williams,carol.w@email.com,+1-555-0103,Chicago,IL,USA,60601,Premium,2019-11-08
C004,David,Brown,david.b@email.com,+1-555-0104,Houston,TX,USA,77001,Standard,2022-01-30
C005,Eva,Davis,eva.davis@email.com,+1-555-0105,Phoenix,AZ,USA,85001,VIP,2018-07-19
C006,Frank,Miller,frank.m@email.com,+1-555-0106,Philadelphia,PA,USA,19101,Standard,2021-09-05
C007,Grace,Wilson,grace.w@email.com,+1-555-0107,San Antonio,TX,USA,78201,Premium,2020-12-14
C008,Henry,Moore,henry.m@email.com,+1-555-0108,San Diego,CA,USA,92101,VIP,2017-04-28
C009,Iris,Taylor,iris.t@email.com,+1-555-0109,Dallas,TX,USA,75201,Standard,2023-02-11
C010,James,Anderson,james.a@email.com,+1-555-0110,San Jose,CA,USA,95101,Premium,2022-08-17


Creating Table with partition Column

In [0]:
%sql
-- Step 1: Create table with partition
CREATE TABLE sales (
    order_id INT,
    city STRING,
    customer STRING,
    amount INT
)
USING DELTA
PARTITIONED BY (city)

****

In [0]:
%sql
OPTIMIZE salesdb.sales
ZORDER BY (customer_id)


path,metrics
abfss://unity-catalog-storage@dbstorageahhzqj5ipen3w.dfs.core.windows.net/7405607761208239/__unitystorage/catalogs/6d40edd1-4147-4b34-bf0b-2194ba125925/tables/8b9fb3eb-b09e-4ef5-b9e6-fb516de4e715,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 4298), 0, List(0, 0), 0, null), 0, 1, 1, false, 0, 0, 1781144263020, 1781144272539, 4, 0, null, List(0, 0), 11, 11, 0, 0, null)"


**Optimize will combine small files into one file for read efficiency and Z-Ordering is a performance optimization technique in Spark (via Delta Lake) that clusters data on disk by columns you frequently filter/join on.**

In [0]:
%sql
OPTIMIZE sales
ZORDER BY (order_id)


path,metrics
abfss://unity-catalog-storage@dbstorageahhzqj5ipen3w.dfs.core.windows.net/7405607761208239/__unitystorage/catalogs/6d40edd1-4147-4b34-bf0b-2194ba125925/tables/4367446c-047f-4835-9a1e-c2c9ae6cd378,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(0, 0), 0, List(0, 0), 0, null), 0, 0, 0, false, 0, 0, 1781142928223, 1781142939820, 4, 0, null, List(0, 0), 4, 4, 0, 0, null)"
